In [2]:
import pandas as pd


df = pd.read_parquet("../data/processed/crimes_raw.parquet")
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

log = []
def record(step):
    log.append({"step": step, "rows_left": len(df)})
    print(f"{step}: {len(df):,} rows")

record("Raw file")
print("Blank dates:", df["Date"].isna().sum())

Raw file: 940,312 rows
Blank dates: 0


In [3]:
import numpy as np
df = df.drop_duplicates(subset="ID")
record("Removed duplicate IDs")

df = df.dropna(subset=["Community Area"])
df["Community Area"] = df["Community Area"].astype(int)
record("Dropped rows with missing Community Area")

df = df.drop(columns=["ID", "Case Number", "Updated On", "Location",
                      "Year", "X Coordinate", "Y Coordinate"], errors="ignore")
record("Dropped unneeded columns")
print(df.columns.tolist())

Removed duplicate IDs: 940,312 rows
Dropped rows with missing Community Area: 940,276 rows
Dropped unneeded columns: 940,276 rows
['Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'Latitude', 'Longitude']


In [4]:
df["IUCR"] = df["IUCR"].astype(str).str.zfill(4)
df["FBI Code"] = df["FBI Code"].astype(str).str.zfill(2)

for col in ["Arrest", "Domestic"]:
    df[col] = df[col].astype(str).str.lower().map({"true": 1, "false": 0})

print(df[["IUCR", "FBI Code", "Arrest", "Domestic"]].head())
print("Arrest values:", df["Arrest"].unique())
print("Domestic values:", df["Domestic"].unique())
print("Blanks:", df[["Arrest", "Domestic"]].isna().sum().tolist())

   IUCR FBI Code  Arrest  Domestic
0  0910       07       0         0
1  0810       06       0         1
2  0910       07       0         0
3  0810       06       0         0
4  5131       26       0         0
Arrest values: [0 1]
Domestic values: [0 1]
Blanks: [0, 0]


In [5]:
import re

rules = [
    ("SCHOOL",      r"SCHOOL|COLLEGE|UNIVERSITY|DAY CARE"),
    ("TRANSPORT",   r"\bCTA\b|\bBUS\b|TRAIN|AIRPORT|AIRCRAFT|TAXI|VEHICLE|RAILROAD|\bL\b|TRANSIT|BRIDGE"),
    ("STREET",      r"STREET|SIDEWALK|ALLEY|HIGHWAY|PARKING LOT|GARAGE|PARK PROPERTY|FOREST PRESERVE|LAKEFRONT|DRIVEWAY"),
    ("COMMERCIAL",  r"STORE|RESTAURANT|BANK|BAR OR|TAVERN|HOTEL|MOTEL|OFFICE|GAS STATION|COMMERCIAL|RETAIL|WAREHOUSE|FACTORY|DEALERSHIP|CURRENCY|SHOP|LAUNDRY|CLUB|GROCERY|MARKET|CASINO|ATM"),
    ("RESIDENTIAL", r"RESIDENCE|APARTMENT|HOUSE|PORCH|YARD|RESIDENTIAL|CHA |HALLWAY|STAIRWELL|BASEMENT|CONDO"),
]

def group_location(x):
    s = str(x).upper()
    for name, pattern in rules:
        if re.search(pattern, s):
            return name
    return "OTHER"

df["location_group"] = df["Location Description"].apply(group_location)

counts = df["location_group"].value_counts()
print(counts)
print("\nShare of OTHER:", round(counts["OTHER"] / len(df) * 100, 1), "%")

location_group
STREET         383414
RESIDENTIAL    317660
COMMERCIAL     143498
TRANSPORT       40601
OTHER           40263
SCHOOL          14840
Name: count, dtype: int64

Share of OTHER: 4.3 %


In [6]:
print(df[df["location_group"] == "OTHER"]["Location Description"].value_counts().head(15))

Location Description
OTHER (SPECIFY)                          15946
HOSPITAL BUILDING / GROUNDS               5718
NURSING / RETIREMENT HOME                 2862
VACANT LOT / LAND                         2792
GOVERNMENT BUILDING / PROPERTY            1852
CHURCH / SYNAGOGUE / PLACE OF WORSHIP     1690
LIBRARY                                   1339
SPORTS ARENA / STADIUM                     713
CONSTRUCTION SITE                          647
ABANDONED BUILDING                         515
CAR WASH                                   436
FEDERAL BUILDING                           221
JAIL / LOCK-UP FACILITY                    153
FIRE STATION                               144
BOAT / WATERCRAFT                          129
Name: count, dtype: int64


In [7]:
loc = df["Location Description"].astype(str).str.upper()

fix = (
    (df["location_group"] == "STREET")
    & loc.str.contains(r"RESIDENCE|RESIDENTIAL|CHA ", regex=True)
    & ~loc.str.contains("NON RESIDENTIAL|NON-RESIDENTIAL", regex=True)
)
print("Rows moved from STREET to RESIDENTIAL:", fix.sum())
print(df[fix]["Location Description"].value_counts().head(8))

df.loc[fix, "location_group"] = "RESIDENTIAL"

print()
print(df["location_group"].value_counts())

Rows moved from STREET to RESIDENTIAL: 13254
Location Description
RESIDENCE - GARAGE           8608
DRIVEWAY - RESIDENTIAL       2971
CHA PARKING LOT / GROUNDS    1667
CHA PARKING LOT                 8
Name: count, dtype: int64

location_group
STREET         370160
RESIDENTIAL    330914
COMMERCIAL     143498
TRANSPORT       40601
OTHER           40263
SCHOOL          14840
Name: count, dtype: int64


In [8]:
violent = ["BATTERY", "ASSAULT", "ROBBERY", "HOMICIDE",
           "CRIMINAL SEXUAL ASSAULT", "WEAPONS VIOLATION"]
df["is_violent"] = df["Primary Type"].isin(violent).astype(int)

top8 = df["Primary Type"].value_counts().head(8).index.tolist()
df["crime_group"] = df["Primary Type"].where(df["Primary Type"].isin(top8), "OTHER")

print("Top 8 crime types:", top8)
print()
print(df["crime_group"].value_counts())
print()
print("Violent share:", round(df["is_violent"].mean() * 100, 1), "%")
print()
print(df.groupby("crime_group")["is_violent"].mean().round(2))

Top 8 crime types: ['THEFT', 'BATTERY', 'CRIMINAL DAMAGE', 'ASSAULT', 'MOTOR VEHICLE THEFT', 'OTHER OFFENSE', 'DECEPTIVE PRACTICE', 'BURGLARY']

crime_group
THEFT                  211907
BATTERY                166062
OTHER                  133846
CRIMINAL DAMAGE        104420
ASSAULT                 83978
MOTOR VEHICLE THEFT     82252
OTHER OFFENSE           61740
DECEPTIVE PRACTICE      59600
BURGLARY                36471
Name: count, dtype: int64

Violent share: 33.4 %

crime_group
ASSAULT                1.00
BATTERY                1.00
BURGLARY               0.00
CRIMINAL DAMAGE        0.00
DECEPTIVE PRACTICE     0.00
MOTOR VEHICLE THEFT    0.00
OTHER                  0.48
OTHER OFFENSE          0.00
THEFT                  0.00
Name: is_violent, dtype: float64


In [ ]:
# 1. rename the catch-all group
df["crime_group"] = df["crime_group"].replace({"OTHER": "ALL OTHER"})
print(df["crime_group"].value_counts())

# 2. cut the last 4 weeks (the newest data is incomplete)
cutoff = (df["Date"].max() - pd.Timedelta(weeks=4)).normalize()
df = df[df["Date"] < cutoff]
record(f"Cut last 4 weeks (kept up to {cutoff.date()})")

# 3. save the clean file and the cleaning log
df.to_parquet("../data/processed/crimes_clean.parquet", index=False)
pd.DataFrame(log).to_csv("../docs/cleaning_log.csv", index=False)

print()
print(pd.DataFrame(log))
print()
print(df.shape)
print(df["Date"].min(), "->", df["Date"].max())

crime_group
THEFT                  211907
BATTERY                166062
ALL OTHER              133846
CRIMINAL DAMAGE        104420
ASSAULT                 83978
MOTOR VEHICLE THEFT     82252
OTHER OFFENSE           61740
DECEPTIVE PRACTICE      59600
BURGLARY                36471
Name: count, dtype: int64
Cut last 4 weeks (kept up to 2026-09-10): 922,290 rows

                                       step  rows_left
0                                  Raw file     940312
1                     Removed duplicate IDs     940312
2  Dropped rows with missing Community Area     940276
3                  Dropped unneeded columns     940276
4  Cut last 4 weeks (kept up to 2026-09-10)     922290

(922290, 18)
2023-01-01 10:19:00 -> 2026-09-10 00:00:00
